# Module 1.1 — The Modern Data Lakehouse & Open Table Formats

**Track:** Big Data Engineering for AI Systems  
**Toolchain:** Delta Lake 4.0 · Apache Iceberg · Apache Hudi · PySpark  
**Objective:** Understand how open table formats bring database-grade reliability to cheap cloud storage, and build a complete Bronze → Silver → Gold Medallion pipeline.

---

## 🧠 What Problem Does the Data Lakehouse Solve?

### The Old World: Pick Your Poison

For years, companies had to choose between two bad options:

| | Data Warehouse | Data Lake |
|---|---|---|
| **Structure** | Rigid schema (SQL tables) | No schema (dump anything) |
| **Cost** | Expensive (proprietary storage) | Cheap (cloud object storage) |
| **Reliability** | ACID transactions ✅ | No transactions ❌ |
| **Flexibility** | Hard to change schema | Store anything, any format |
| **Problem** | Too expensive for AI-scale data | Becomes a "data swamp" — nobody trusts it |

### The Lakehouse: Best of Both Worlds

A **data lakehouse** adds a **metadata layer** on top of cheap cloud storage (S3, GCS, ADLS). This metadata layer provides:
- **ACID transactions** — no partial writes, no corrupted reads
- **Schema enforcement** — the system rejects bad data before it enters
- **Time travel** — query any previous version of your data
- **Concurrent access** — multiple writers don't corrupt each other

**Analogy:** A data lake is like a parking lot where anyone dumps cars anywhere. A data lakehouse is the same parking lot, but now with numbered spots, security cameras, and a log of every car that entered/left.

---

## 📚 The Four Open Table Formats (2026)

| Format | Philosophy | Best For | Who Uses It |
|--------|-----------|---------|-------------|
| **Apache Iceberg** | Engine-neutral, massive scale | Cross-engine queries, exabyte datasets | Netflix, Apple, Airbnb |
| **Delta Lake 4.0** | Spark-native, simplicity | Unified batch+streaming, Medallion arch | Databricks, most Spark shops |
| **Apache Hudi** | Streaming-first, fast upserts | Near real-time CDC pipelines | Uber, Amazon |
| **Apache Paimon** | LSM-tree, stream-native | Unified batch-stream at event layer | Alibaba, Flink ecosystem |

### 🤔 Which Format Should I Choose?

| If you need... | Use |
|---|---|
| Query from Spark AND Trino AND Flink | **Iceberg** (engine-neutral) |
| Simplest Spark integration | **Delta Lake** (Spark-native) |
| Real-time upserts from CDC streams | **Hudi** (record-level indexing) |
| Pure streaming data lake | **Paimon** (LSM compaction) |
| You're unsure | **Delta Lake** (easiest to start, UniForm reads as Iceberg) |

### 🤔 What is ACID and Why Does It Matter for AI?

**A**tomicity: A write either fully succeeds or fully fails (no half-written files)  
**C**onsistency: Data always matches the defined schema  
**I**solation: Two jobs writing simultaneously don't corrupt each other  
**D**urability: Once written, data survives crashes  

**Without ACID:** Your training pipeline reads a dataset while another job is writing to it → model trains on corrupted data → deployed model makes wrong predictions → revenue loss.

**With ACID:** Readers always see a complete, consistent snapshot. Writers queue safely.

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors treat data as static files (CSV/JSON). Seniors treat data as a dynamic, versioned, ACID-compliant database. You cannot build reliable AI pipelines on top of messy file systems; you need a Data Lakehouse.

**Mental Model:** A Data Lakehouse (Iceberg/Delta) combines the infinite cheap storage of a 'Data Lake' (S3) with the safety, transactions, and speed of a 'SQL Database'.

**Common Junior Pitfall:** Writing raw Parquet files directly to S3 and trying to query them, leading to schema mismatches, corrupted data states during write failures, and unmanageable small-file problems.

---


In [ ]:
# Cell 1 — Install dependencies
!pip install -q delta-spark pyspark pandas pyarrow

---

## 1 · Delta Lake 4.0: The Medallion Architecture

### 🤔 What is the Medallion Architecture?

It's a **3-layer data quality pattern**:

```
Raw data → [Bronze] → [Silver] → [Gold]
              │           │           │
          Raw ingestion  Cleaned    Aggregated
          (as-is)       (typed)    (business-ready)
```

| Layer | Data Quality | Schema | Who Uses It |
|-------|-------------|--------|-------------|
| **Bronze** | Raw, may have errors | Schema-on-read | Data engineers (debugging) |
| **Silver** | Cleaned, typed, deduplicated | Schema-on-write | Data scientists (exploration) |
| **Gold** | Aggregated, business metrics | Strict | ML models, dashboards, executives |

### 🤔 Why 3 layers? Why not clean data in one step?

1. **Debugging:** When a model gives wrong predictions, you trace back: Gold → Silver → Bronze → find the raw error
2. **Reprocessing:** If your cleaning logic was wrong, re-run Silver from Bronze (raw data is never lost)
3. **Multiple consumers:** Different teams need different aggregations from the same Silver data

In [ ]:
# Cell 2 — Build a Medallion pipeline with Delta Lake
#
# We simulate sensor data from IoT devices (e.g., factory machines).
# This is a common AI use case: predict equipment failure from sensor readings.

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n_records = 5000

# Simulate raw sensor data (with realistic messiness)
raw_data = pd.DataFrame({
    'device_id': np.random.choice(['sensor_A1', 'sensor_B2', 'sensor_C3', None, ''], n_records, p=[0.3, 0.3, 0.3, 0.05, 0.05]),
    'temperature': np.where(np.random.random(n_records) > 0.95, np.nan, np.random.normal(72, 8, n_records)),
    'vibration': np.where(np.random.random(n_records) > 0.97, -999, np.random.exponential(2.5, n_records)),
    'pressure_psi': np.random.normal(14.7, 1.2, n_records),
    'timestamp': [datetime(2026, 1, 1) + timedelta(minutes=i*5 + np.random.randint(-2, 3)) for i in range(n_records)],
    'raw_log': ['OK'] * (n_records - 50) + ['WARN: high temp'] * 30 + ['ERROR: sensor offline'] * 20,
})

np.random.shuffle(raw_data['raw_log'].values)

print('=== BRONZE LAYER (Raw Ingestion) ===')
print(f'  Records:        {len(raw_data):,}')
print(f'  Null device_id: {raw_data["device_id"].isna().sum() + (raw_data["device_id"] == "").sum()}')
print(f'  Null temperature: {raw_data["temperature"].isna().sum()}')
print(f'  Invalid vibration (-999): {(raw_data["vibration"] == -999).sum()}')
print(f'\n  Sample (notice the messy data):')
print(raw_data.head(10).to_string(index=False))

In [ ]:
# Cell 3 — Silver layer: clean, type, deduplicate
#
# WHAT IS HAPPENING?
# 1. Drop rows with missing device_id (can't identify the source)
# 2. Replace sentinel values (-999) with NaN
# 3. Fill NaN temperatures with device-specific median (not global!)
# 4. Add computed features useful for downstream ML
# 5. Enforce strict data types

silver = raw_data.copy()

# Step 1: Remove rows with missing/empty device_id
silver = silver[silver['device_id'].notna() & (silver['device_id'] != '')]

# Step 2: Replace sentinel values
silver['vibration'] = silver['vibration'].replace(-999, np.nan)

# Step 3: Fill nulls with per-device median (smarter than global median)
silver['temperature'] = silver.groupby('device_id')['temperature'].transform(
    lambda x: x.fillna(x.median())
)
silver['vibration'] = silver.groupby('device_id')['vibration'].transform(
    lambda x: x.fillna(x.median())
)

# Step 4: Feature engineering for ML
silver['temp_zscore'] = silver.groupby('device_id')['temperature'].transform(
    lambda x: (x - x.mean()) / x.std()
)
silver['is_anomaly'] = (silver['temp_zscore'].abs() > 2) | (silver['vibration'] > 8)

# Step 5: Strict types
silver['timestamp'] = pd.to_datetime(silver['timestamp'])

print('=== SILVER LAYER (Cleaned) ===')
print(f'  Records: {len(silver):,} (dropped {len(raw_data) - len(silver)} bad rows)')
print(f'  Nulls remaining: {silver.isna().sum().sum()}')
print(f'  Anomalies detected: {silver["is_anomaly"].sum()}')
print(f'\n  Sample (clean, typed, enriched):')
print(silver.head(8).to_string(index=False))

In [ ]:
# Cell 4 — Gold layer: aggregated business-ready features
#
# Gold tables serve two consumers:
# 1. ML models (features like rolling averages, anomaly rates)
# 2. Dashboards (KPIs for executives)

gold = silver.set_index('timestamp').groupby('device_id').resample('1h').agg(
    avg_temp=('temperature', 'mean'),
    max_temp=('temperature', 'max'),
    avg_vibration=('vibration', 'mean'),
    max_vibration=('vibration', 'max'),
    anomaly_count=('is_anomaly', 'sum'),
    reading_count=('temperature', 'count'),
).reset_index()

gold['anomaly_rate'] = gold['anomaly_count'] / gold['reading_count']
gold['health_score'] = (1 - gold['anomaly_rate']).clip(0, 1) * 100

print('=== GOLD LAYER (ML-Ready Features) ===')
print(f'  Records: {len(gold):,} (1-hour aggregations)')
print(f'  Devices: {gold["device_id"].nunique()}')
print(f'\n  Sample (ready for ML training):')
print(gold.head(10).to_string(index=False))
print(f'\n  Avg health score: {gold["health_score"].mean():.1f}%')

---

## 2 · Time Travel & Schema Evolution

### 🤔 What is Time Travel?

Every write to a Delta/Iceberg table creates a **version snapshot**. You can query ANY previous version:

```sql
-- Query the table as it was yesterday
SELECT * FROM sensors VERSION AS OF 42;
SELECT * FROM sensors TIMESTAMP AS OF '2026-02-28 14:00:00';
```

**Why this matters for AI:**
- **Reproducibility:** "What data did model v3 train on?" → Query the exact version
- **Debugging:** "Why did predictions break on March 1?" → Compare data snapshots before/after
- **Compliance:** EU AI Act requires data provenance → time travel provides it

### 🤔 What is Schema Evolution?

Your data schema WILL change. New sensors get added, column names get renamed. Without schema evolution:
- Option A: Rewrite ALL historical data (expensive, risky)
- Option B: Create a new table (breaks all downstream pipelines)

With schema evolution, the table format **merges** old and new schemas automatically:
```
v1: [device_id, temperature, pressure]
v2: [device_id, temperature, pressure, humidity]  ← new column added
v3: [device_id, temp_celsius, pressure, humidity]  ← column renamed
```
Old data files are NOT rewritten. The metadata layer handles the mapping.

In [ ]:
# Cell 5 — Simulate time travel and schema evolution
import copy

class SimulatedDeltaTable:
    """Simulates Delta Lake version history for educational purposes."""
    def __init__(self):
        self.versions = []
        self.current_version = -1

    def write(self, df, description):
        self.current_version += 1
        self.versions.append({
            'version': self.current_version,
            'data': df.copy(),
            'columns': list(df.columns),
            'rows': len(df),
            'description': description,
            'timestamp': datetime.utcnow().isoformat(),
        })
        print(f'  v{self.current_version}: {description} ({len(df)} rows, cols={list(df.columns)})')

    def read_version(self, version):
        return self.versions[version]['data']

# Build version history
table = SimulatedDeltaTable()
print('Writing versions (time travel history):')

# v0: Initial bronze
table.write(raw_data[['device_id', 'temperature', 'pressure_psi']], 'Initial sensor data')

# v1: Add vibration column (schema evolution)
v1_data = raw_data[['device_id', 'temperature', 'pressure_psi', 'vibration']].head(3000)
table.write(v1_data, 'Added vibration column')

# v2: Clean data (silver)
table.write(silver[['device_id', 'temperature', 'pressure_psi', 'vibration', 'is_anomaly']], 'Cleaned + anomaly flag')

# Time travel: compare versions
print(f'\nTime Travel Demo:')
for v in range(len(table.versions)):
    df = table.read_version(v)
    print(f'  v{v}: {table.versions[v]["description"]}')
    print(f'       {df.shape[0]} rows x {df.shape[1]} cols = {list(df.columns)}')

---

## 3 · Delta UniForm: Cross-Engine Interoperability

### 🤔 What is UniForm and Why Does It Matter?

The biggest risk of choosing a table format is **vendor lock-in**. If you build everything on Delta Lake, can Trino or Flink read your tables?

**Delta UniForm** solves this by writing **dual metadata**: Delta log files + Iceberg manifests simultaneously.

```
Write data as Delta Lake
       │
       ├── Delta clients read via _delta_log/
       ├── Iceberg clients read via metadata/
       └── Hudi clients read via .hoodie/
```

| Without UniForm | With UniForm |
|---|---|
| Each engine needs its own copy | One copy, all engines read it |
| Storage costs 3x | Storage costs 1x |
| ETL to convert between formats | Zero conversion needed |

### 🤔 When NOT to Use UniForm?
- When you only use Spark (no cross-engine need)
- When you need Iceberg-specific features like hidden partitioning
- Small datasets where the metadata overhead isn't justified

In [ ]:
# Cell 6 — UniForm configuration example

uniform_config = '''
-- Enable Delta UniForm when creating a table
CREATE TABLE sensor_metrics (
    device_id STRING,
    avg_temp DOUBLE,
    anomaly_rate DOUBLE,
    reading_date DATE
)
USING delta
TBLPROPERTIES (
    'delta.universalFormat.enabledFormats' = 'iceberg,hudi',
    'delta.enableDeletionVectors' = 'true',
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7'
);

-- Now this SAME table can be read by:
-- Spark:  spark.read.format('delta').load('s3://bucket/sensor_metrics')
-- Trino:  SELECT * FROM iceberg.default.sensor_metrics  -- reads Iceberg metadata!
-- Flink:  SELECT * FROM hudi_catalog.sensor_metrics     -- reads Hudi metadata!
'''
print('Delta UniForm SQL Configuration:')
print(uniform_config)

---

## 4 · Compaction & Snapshot Retention

### 🤔 What is Compaction and Why Is It Needed?

Every write adds small files to storage. After thousands of writes, you have millions of tiny files. Queries slow down because they must open each file individually.

**Compaction** merges small files into larger ones:
```
Before: file1.parquet (2MB), file2 (1MB), file3 (3MB), ... file500 (1MB)
After:  merged_1.parquet (256MB), merged_2.parquet (256MB)
```

### ⚖️ Compaction Trade-offs

| Setting | Low | High |
|---------|-----|------|
| Compaction frequency | Faster writes, slower reads | Slower writes, faster reads |
| Target file size | More parallelism | Less file overhead |
| Snapshot retention | Less storage | More time travel history |

**Rule of thumb:** Compact every 1-4 hours. Target 256MB–1GB files. Keep 7-30 days of snapshots.

In [ ]:
# Cell 7 — Simulate compaction decisions

# Simulate the effect of compaction on query performance
file_sizes_before = np.random.exponential(5, 500)  # 500 small files, avg 5MB
target_size = 256  # MB

total_data = file_sizes_before.sum()
n_files_after = max(1, int(total_data / target_size))

print('Compaction Analysis:')
print(f'  Before: {len(file_sizes_before)} files, avg {file_sizes_before.mean():.1f} MB, total {total_data:.0f} MB')
print(f'  After:  {n_files_after} files, avg {target_size} MB, total {total_data:.0f} MB')
print(f'  File reduction: {len(file_sizes_before)/n_files_after:.0f}x fewer files')
print(f'  Query speedup:  ~{len(file_sizes_before)/n_files_after:.0f}x (fewer file opens)')

# Snapshot retention policy
print(f'\nSnapshot Retention Policy:')
retention_days = 30
versions_per_day = 24  # hourly writes
total_snapshots = retention_days * versions_per_day
print(f'  Retention: {retention_days} days')
print(f'  Estimated snapshots: {total_snapshots}')
print(f'  Storage overhead: ~{total_snapshots * 0.1:.0f} MB metadata')
print(f'  After {retention_days} days: VACUUM removes expired snapshots')

---

## Production Reality Check

### What We Simulated vs What Production Looks Like

| In This Notebook | In Production |
|---|---|
| `SimulatedDeltaTable` class | Real Delta Lake on S3/ADLS with Spark cluster |
| Pandas DataFrames | PySpark DataFrames (distributed across 100+ machines) |
| In-memory version history | `_delta_log/` JSON files on cloud storage |
| Single-machine execution | Multi-node cluster with driver + workers |

### Real Production Setup (Estimated Effort)

| Component | Tool | Setup Time | Managed Alternative |
|-----------|------|-----------|--------------------|
| Object storage | AWS S3 / Azure ADLS | 1 hour | Already exists in cloud |
| Spark cluster | EMR / Databricks / Dataproc | 2-4 hours | Databricks (minutes) |
| Delta Lake config | `spark.conf.set("spark.sql.extensions", ...)` | 30 min | Built into Databricks |
| Compaction jobs | Scheduled OPTIMIZE commands | 1-2 hours | Auto-optimize in managed |
| Monitoring | Delta Lake metrics + Prometheus | 2-4 hours | Built into managed |

**Key insight:** Most teams start with managed Databricks (5-minute setup). Self-hosting is only needed for extreme cost optimization or data residency requirements.


---

## 🎯 Summary & Key Takeaways

| Concept | What You Learned | Why It Matters for AI |
|---------|-----------------|---------------------|
| Data Lakehouse | Database reliability on cheap storage | Trustworthy training data |
| Medallion Architecture | Bronze → Silver → Gold quality layers | Traceable data lineage |
| Open Table Formats | Iceberg, Delta, Hudi, Paimon | Choose the right tool |
| Time Travel | Query any historical version | Reproduce training datasets |
| Schema Evolution | Add/rename columns without rewriting | Evolve data without breaking pipelines |
| UniForm | One table, all engines can read | No vendor lock-in |
| Compaction | Merge small files for query speed | Operational efficiency |

**Next →** `03_data_management.ipynb` — Data Versioning & Validation